# Build dbt Models on Opteryx.app

Runs [dbt](https://www.getdbt.com) against [opteryx.app](https://opteryx.app), the hosted service, with the `dbt-opteryx` adapter. It builds a small project over two public datasets, the **National Vulnerability Database (NVD)** and CISA's catalog of **known-exploited vulnerabilities**: staging views, two tables, data tests and a unit test. Then it reads the tables it built back into pandas.

This notebook accompanies [dbt](https://docs.opteryx.app/docs/integrations/dbt) on docs.opteryx.app.

**You need an opteryx.app account.** dbt runs every statement through the Jobs API, which needs a credential. Signing up is free at [opteryx.app](https://opteryx.app). Then create a personal access token in Opteryx Studio under **Settings → API Tokens**. It gives you a **client ID** and a **client secret**, and the secret is shown only once.

dbt writes its models into your personal space, `personal.<your client ID>`. The last cell drops them again.

## 1. Install

In [ ]:
%pip install -q dbt-opteryx pandas matplotlib

## 2. Credentials

Keep the client secret out of the notebook itself, so it isn't saved with it or shared by accident. This cell looks for the client ID and secret in three places, in order:

1. **Colab Secrets.** Click the key icon in Colab's left sidebar, add `OPTERYX_CLIENT_ID` and `OPTERYX_CLIENT_SECRET`, and turn on notebook access for both. Secrets stay with your Colab account, so you set them once and every notebook can use them.
2. **Environment variables** with the same names, for running the notebook outside Colab.
3. **A prompt.** If neither is set, the cell asks, and the secret is typed into a hidden field.

The values are put into environment variables, which is where the dbt profile below reads them from.

In [ ]:
import os
from getpass import getpass


def credential(name, secret=False):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        pass   # not in Colab, or the secret isn't set or shared with this notebook
    if os.environ.get(name):
        return os.environ[name]
    return getpass(f"{name}: ") if secret else input(f"{name}: ")


os.environ["OPTERYX_CLIENT_ID"] = credential("OPTERYX_CLIENT_ID")
os.environ["OPTERYX_CLIENT_SECRET"] = credential("OPTERYX_CLIENT_SECRET", secret=True)
print("Client ID:", os.environ["OPTERYX_CLIENT_ID"])

## 3. Create the dbt project

A dbt project is a folder of YAML and SQL files. These cells write one into `nvd_dbt/`, one file per cell.

`profiles.yml` holds the connection. dbt's `database` is an Opteryx **workspace** and its `schema` is a **collection**, so models are built in `personal.<your client ID>`. `user` and `token` carry the client ID and secret.

In [ ]:
for folder in ("nvd_dbt/models", "nvd_dbt/macros"):
    os.makedirs(folder, exist_ok=True)

In [ ]:
%%writefile nvd_dbt/profiles.yml
nvd_dbt:
  target: cloud
  outputs:
    cloud:
      type: opteryx
      host: opteryx.app
      port: 443
      ssl: true
      user: "{{ env_var('OPTERYX_CLIENT_ID') }}"
      token: "{{ env_var('OPTERYX_CLIENT_SECRET') }}"
      database: personal
      schema: "{{ env_var('OPTERYX_CLIENT_ID') }}"
      threads: 1

`dbt_project.yml` names the project. `query-comment: append: true` moves the comment dbt adds to every statement to the end of it, because Opteryx doesn't yet accept one in front of `CREATE COLLECTION`. Models are views unless they say otherwise.

In [ ]:
%%writefile nvd_dbt/dbt_project.yml
name: nvd_dbt
version: "1.0"
profile: nvd_dbt
query-comment:
  append: true
models:
  nvd_dbt:
    +materialized: view

`sources.yml` declares the two public datasets as dbt sources, so models refer to them with `source()` rather than a hard-coded name. dbt can test sources too: every CISA entry must have a CVE ID, and no CVE may appear twice.

In [ ]:
%%writefile nvd_dbt/models/sources.yml
version: 2
sources:
  - name: security
    database: public
    schema: security
    tables:
      - name: nvd_vulnerabilities
      - name: cisa_kev
        columns:
          - name: cve_id
            data_tests:
              - not_null
              - unique

Two staging views pick the columns the rest of the project uses:

In [ ]:
%%writefile nvd_dbt/models/stg_cves.sql
SELECT cve_id, published_at, severity, cvss_score
  FROM {{ source('security', 'nvd_vulnerabilities') }}

In [ ]:
%%writefile nvd_dbt/models/stg_exploited.sql
SELECT cve_id, vendor_project, product, date_added, known_ransomware_campaign_use
  FROM {{ source('security', 'cisa_kev') }}

`exploited_cves` is a table: each known-exploited CVE with its NVD severity and score. `ref()` points at the staging views, which is how dbt knows to build them first. Each column is named with `AS`: Opteryx names a qualified column such as `kev.cve_id` with its table alias, and the models built on this one refer to plain `cve_id`.

In [ ]:
%%writefile nvd_dbt/models/exploited_cves.sql
{{ config(materialized='table') }}
SELECT kev.cve_id                        AS cve_id,
       kev.vendor_project                AS vendor_project,
       kev.product                       AS product,
       kev.date_added                    AS date_added,
       kev.known_ransomware_campaign_use AS known_ransomware_campaign_use,
       nvd.severity                      AS severity,
       nvd.cvss_score                    AS cvss_score
  FROM {{ ref('stg_exploited') }} AS kev
 INNER JOIN {{ ref('stg_cves') }} AS nvd
    ON kev.cve_id = nvd.cve_id

In [ ]:
%%writefile nvd_dbt/models/exploited_per_vendor.sql
{{ config(materialized='table') }}
SELECT vendor_project,
       COUNT(*) AS exploited_cves,
       SUM(CASE WHEN known_ransomware_campaign_use = 'Known' THEN 1 ELSE 0 END) AS used_by_ransomware
  FROM {{ ref('exploited_cves') }}
 GROUP BY vendor_project

`schema.yml` adds data tests to the models, and a **unit test**. Data tests check the built tables. The unit test checks the logic of `exploited_per_vendor` against three made-up rows, so it doesn't depend on what is in the data today.

In [ ]:
%%writefile nvd_dbt/models/schema.yml
version: 2
models:
  - name: exploited_cves
    columns:
      - name: cve_id
        data_tests:
          - not_null
          - unique
          - relationships:
              arguments:
                to: ref('stg_exploited')
                field: cve_id
      - name: severity
        data_tests:
          - accepted_values:
              arguments:
                values: ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'NONE']
  - name: exploited_per_vendor
    columns:
      - name: vendor_project
        data_tests:
          - unique

unit_tests:
  - name: counts_cves_and_ransomware_per_vendor
    model: exploited_per_vendor
    given:
      - input: ref('exploited_cves')
        rows:
          - {cve_id: CVE-1, vendor_project: Acme, known_ransomware_campaign_use: Known}
          - {cve_id: CVE-2, vendor_project: Acme, known_ransomware_campaign_use: Unknown}
          - {cve_id: CVE-3, vendor_project: Initech, known_ransomware_campaign_use: Unknown}
    expect:
      rows:
        - {vendor_project: Acme, exploited_cves: 2, used_by_ransomware: 1}
        - {vendor_project: Initech, exploited_cves: 1, used_by_ransomware: 0}

A macro for the clean-up cell at the end. It drops every model this project built:

In [ ]:
%%writefile nvd_dbt/macros/drop_models.sql
{% macro drop_models() %}
  {% for node in graph.nodes.values() if node.resource_type == 'model' %}
    {% set relation = adapter.get_relation(database=node.database, schema=node.schema, identifier=node.alias) %}
    {% if relation %}
      {% do adapter.drop_relation(relation) %}
      {{ log('Dropped ' ~ relation, info=True) }}
    {% endif %}
  {% endfor %}
{% endmacro %}

## 4. Run dbt

`dbt()` runs a dbt command against the project, the same as typing `dbt ...` in a terminal, and raises if it fails. `dbt debug` checks the profile and the connection. If it fails on the connection, opteryx.app didn't accept the client ID and secret: check both values, and that the token hasn't been revoked.

In [ ]:
from dbt.cli.main import dbtRunner


def dbt(*args):
    result = dbtRunner().invoke([*args, "--project-dir", "nvd_dbt", "--profiles-dir", "nvd_dbt"])
    if not result.success:
        raise RuntimeError(f"dbt {' '.join(args)} failed")


dbt("debug")

`dbt run` builds the four models in dependency order: the two views, then `exploited_cves`, then `exploited_per_vendor`.

In [ ]:
dbt("run")

`dbt test` runs the source tests, the data tests and the unit test.

In [ ]:
dbt("test")

`dbt show` prints rows from a model:

In [ ]:
dbt("show", "--select", "exploited_per_vendor", "--limit", "5")

## 5. Read the tables back

The models are ordinary datasets on opteryx.app, so anything that queries opteryx.app can read them. `dbt-opteryx` installed `opteryx-sqlalchemy`, so pandas can read them directly. The vendors with the most known-exploited CVEs:

In [ ]:
import pandas as pd
from sqlalchemy import URL, create_engine

engine = create_engine(
    URL.create(
        "opteryx",
        username=os.environ["OPTERYX_CLIENT_ID"],
        password=os.environ["OPTERYX_CLIENT_SECRET"],
        host="opteryx.app",
        port=443,
        database="default",
        query={"ssl": "true"},
    )
)
models = f"personal.{os.environ['OPTERYX_CLIENT_ID']}"

with engine.connect() as conn:
    vendors = pd.read_sql_query(
        f"SELECT * FROM {models}.exploited_per_vendor ORDER BY exploited_cves DESC LIMIT 15", conn
    )
vendors.plot.barh(x="vendor_project", y=["exploited_cves", "used_by_ransomware"], figsize=(10, 6),
                  title="Known-exploited CVEs per vendor").invert_yaxis()

And how severe the known-exploited CVEs are, by NVD severity:

In [ ]:
with engine.connect() as conn:
    by_severity = pd.read_sql_query(
        f"SELECT severity, COUNT(*) AS cves FROM {models}.exploited_cves GROUP BY severity ORDER BY cves DESC",
        conn,
    )
by_severity

## 6. Clean up

This drops the four models from your personal space. Skip it if you want to keep them.

In [ ]:
dbt("run-operation", "drop_models")

## Things to know

- **Every statement is a query.** dbt sends several statements per model, including the ones it uses to look up what already exists, and each one counts towards your account's query allowance.
- **No transactions.** Every statement commits on its own, so a model that fails halfway can leave its earlier statements applied.
- **Snapshots aren't supported**, and **model contracts** are refused with an error naming the model.
- The [dbt integration page](https://docs.opteryx.app/docs/integrations/dbt) lists what works and the known limitations.